# Etapa 4 — Activación y movimiento individual

**Objetivo:** comprobar los métodos de `Agente` sin ejecutar toda la simulación. Se revisan la propuesta de movimiento, la activación probabilística, las prioridades BFS y humo, el escape de una celda bloqueada y el registro de la evacuación.

**Antes de ejecutar:** completar los métodos pendientes de `Agente` y el BFS de la etapa anterior.

## Prueba 1. Proponer no es moverse

En ausencia de humo y desvíos, desde `(2, 7)` debe proponerse `(1, 7)`, que está más cerca de la salida. La propuesta no cambia la posición, pero `mover()` sí.

In [1]:
import numpy as np
from modelo import Agente, Configuracion, Entorno

# Sin humo ni desvíos, se aísla la decisión según BFS.
entorno = Entorno(Configuracion(usar_humo=False, probabilidad_desvio=0))
distancias = entorno.calcular_distancias()
agente = Agente(0, 2, 7, "alta", 1.0)

assert agente.esta_habilitado(np.random.default_rng(1))

# La celda (1, 7) queda un paso más cerca de la salida.
destino = agente.elegir_destino(
    entorno, {(2, 7): 0}, distancias, set(), np.random.default_rng(3)
)
assert destino == (1, 7)

# Elegir no mueve al agente; mover() sí modifica su posición.
assert agente.posicion() == (2, 7)
agente.mover(destino)
assert agente.posicion() == (1, 7)
print("Prueba 1: propuesta y movimiento correctos.")

NotImplementedError: Completar elegir_destino

## Prueba 2. Activación probabilística

En 1000 sorteos, un agente con probabilidad 1,0 debe activarse todas las veces. Con probabilidad 0,7 se esperan **aproximadamente 700 activaciones**, no exactamente 700; por eso se acepta un intervalo amplio.

In [ ]:
alta = Agente(1, 5, 5, "alta", 1.0)
baja = Agente(2, 5, 6, "baja", 0.7)
aleatorio = np.random.default_rng(41)

# Repetir el sorteo y contar las activaciones.
cuenta_alta = sum(alta.esta_habilitado(aleatorio) for _ in range(1000))
cuenta_baja = sum(baja.esta_habilitado(aleatorio) for _ in range(1000))
assert cuenta_alta == 1000
assert 620 < cuenta_baja < 780
print("Prueba 2: activaciones", cuenta_alta, cuenta_baja)

## Prueba 3. Prioridad BFS y humo

Cada candidato tiene formato `(distancia_BFS, humo, posición)`. Primero debe elegirse la **menor distancia** y, entre opciones empatadas, el **menor humo**. La opción esperada es `(9, 9)`.

In [ ]:
opciones = [
    (4, 0.0, (8, 8)),  # Menos humo, pero más lejos.
    (3, 0.8, (9, 8)),  # Más cerca, pero mucho humo.
    (3, 0.2, (9, 9)),  # Más cerca y menos humo.
]
mejor = agente._seleccionar_mejor_opcion(opciones, np.random.default_rng(7))
assert mejor == (9, 9)
print("Prueba 3: prioridades correctas.")

## Prueba 4. Abandonar una celda bloqueada

Se coloca artificialmente humo 0,7 en `(2, 7)`, ocupada por un agente. Aunque esté bloqueada para nuevos ingresos, el agente debe poder **proponer salir** a un vecino transitable con ruta. Proponer no altera todavía su posición.

In [ ]:
entorno_humo = Entorno(Configuracion())
entorno_humo.humo[7, 2] = 0.7
agente_bloqueado = Agente(3, 2, 7, "baja", 0.7)

# Obtener una propuesta desde una posición bloqueada.
propuesta = agente_bloqueado.elegir_destino(
    entorno_humo, {(2, 7): 3},
    entorno_humo.calcular_distancias(), set(), np.random.default_rng(5)
)
assert propuesta is not None
assert entorno_humo.es_transitable(*propuesta)
assert agente_bloqueado.posicion() == (2, 7)
print("Prueba 4: propuesta de salida válida.")

## Prueba 5. Registrar la evacuación

El método `evacuar()` debe registrar que la persona salió, **qué salida utilizó** y **en qué paso**. Un agente evacuado no puede volver a activarse.

In [ ]:
agente_evac = Agente(4, 1, 7, "alta", 1.0)
agente_evac.evacuar("izquierda", 3)

assert agente_evac.evacuado
assert agente_evac.salida_utilizada == "izquierda"
assert agente_evac.tiempo_evacuacion == 3
assert not agente_evac.esta_habilitado(np.random.default_rng(9))
print("Prueba 5: evacuación registrada.")

**Para discutir:** ¿por qué se separan la activación, la propuesta y la ejecución del movimiento?